# Entradas a Almacén (SGA) → Dataset

Importa las recepciones de mercancía del endpoint canónico `entradas_almacen`
(`POST /galileo/lisa/geninf/isp/sga_entradas`) de la API LISA y las carga en un
`DataFrame` de pandas listo para analizar.

**Requisitos**

- Token en la variable de entorno `LISA_TOKEN` o en un `.env` en la raíz del proyecto.
- Dependencias: `uv pip install -r requirements.txt` (incluye `pandas` y `jupyter`).

**Convenciones LISA**

- `fecha_gen` en formato `dd/mm/aaaa`.
- "Todos los registros" → `regs=-1` (default). **Nunca** `pag=-1`.

In [7]:
import os

import pandas as pd

from lisa_client import LisaClient, LisaAPIError

# El token se lee de LISA_TOKEN (.env o variable de entorno) dentro de LisaClient.
assert os.getenv("LISA_TOKEN"), (
    "Falta LISA_TOKEN. Expórtalo o ponlo en un .env en la raíz del proyecto."
)

## Parámetros de la consulta

Ajusta `FECHA_GEN` (obligatorio) y, opcionalmente, `ALMACEN` (vacío = todos los almacenes).

In [8]:
FECHA_GEN = "01/06/2026"  # dd/mm/aaaa
ALMACEN = "5460D"        # "" = todos los almacenes
REGS = -1                  # -1 = todos los registros (default canónico)

## Llamada al API

In [9]:
client = LisaClient()

try:
    respuesta = client.entradas_almacen(
        fecha_gen=FECHA_GEN,
        almacen=ALMACEN,
        regs=REGS,
    )
except LisaAPIError as e:
    raise SystemExit(f"Error en la API LISA: {e}")

rowstat = respuesta.get("registros", {}).get("rowstat", {})
print("rowstat:", rowstat)

17:22:07 [INFO] POST /galileo/lisa/geninf/isp/sga_entradas | body: {'p_fecha_gen': '01/06/2026', 'p_almacen': '5460D'}


rowstat: {'numero_registros': 483}


## Construcción del dataset

Los registros de negocio vienen en `registros.reg`. Cuando hay un solo registro la
API devuelve un `dict` en lugar de una lista, así que lo normalizamos antes de
pasarlo a pandas.

In [10]:
registros = respuesta.get("registros", {})
reg = registros.get("reg") or []
if isinstance(reg, dict):  # un solo registro -> lista de uno
    reg = [reg]

df = pd.json_normalize(reg)
print(f"Filas: {len(df)}  |  Columnas: {df.shape[1]}")
df.head()

Filas: 483  |  Columnas: 34


,numero_tarea,fecha_gen,usuario_gen,maquina_gen,fecha_inicio,fecha_fin,prioridad,maquina,usuario,status_tarea,...,zona_almacen,situacion,zona_trabajo,tipo_entrada,tipo_recepcion_paletizada,entrada_sga,tratamiento_especial,reserva_ped_vtas,trabajador,numero_factura_fiscal
0,13920,01/06/2026,GAPEAR,XO,01/06/2026,01/06/2026,50,XO,GAPEAR,C,...,02,CAL,RC,ES,D,S,N,S,GAPEAR,NaN
1,14028,04/06/2026,SOCALEG,RB,04/06/2026,04/06/2026,50,RB,SOCALEG,C,...,02,CAL,RC,ES,D,S,N,S,SOCALEG,NaN
2,14033,04/06/2026,SELANDE,RD,04/06/2026,04/06/2026,50,RD,SELANDE,C,...,02,CAL,RC,ES,D,S,N,S,SELANDE,NaN
3,14041,04/06/2026,CAVAVA,RE,04/06/2026,04/06/2026,50,RE,CAVAVA,C,...,02,CAL,RC,ES,D,S,N,S,CAVAVA,NaN
4,14067,05/06/2026,ENSAPAOD,XO,05/06/2026,05/06/2026,50,XO,ENSAPAOD,C,...,02,CAL,RC,ES,D,S,N,S,ENSAPAOD,NaN


In [11]:
# Inventario de columnas y tipos inferidos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 483 entries, 0 to 482
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   numero_tarea               483 non-null    int64
 1   fecha_gen                  483 non-null    str  
 2   usuario_gen                483 non-null    str  
 3   maquina_gen                483 non-null    str  
 4   fecha_inicio               483 non-null    str  
 5   fecha_fin                  482 non-null    str  
 6   prioridad                  483 non-null    int64
 7   maquina                    483 non-null    str  
 8   usuario                    483 non-null    str  
 9   status_tarea               483 non-null    str  
 10  tipo_movimiento            483 non-null    str  
 11  numero_doc_interno         483 non-null    str  
 12  proveedor                  483 non-null    str  
 13  numero_doc_externo         483 non-null    str  
 14  fecha_doc_externo          483 non-nu

## Persistir el dataset (opcional)

Guarda una copia en CSV para análisis posterior.

In [12]:
salida = f"entradas_{FECHA_GEN.replace('/', '-')}_{ALMACEN or 'todos'}.csv"
df.to_csv(salida, index=False, encoding="utf-8")
print(f"Guardado en {salida}")

Guardado en entradas_01-06-2026_5460D.csv
